# Il modello intero: dal numero del carattere alla previsione

*Anatomia di un LLM* · capitolo 5, L'attenzione e il blocco · script `03-10`, `scripts/03_10_modello_completo.py`

Tutti i pezzi sono pronti. Qui si mettono in fila e si verifica che il risultato
sia un modello vero: che la forma non cambi mai dentro la pila, che non sappia
nulla del futuro, e che appena costruito sia esattamente ignorante quanto deve.

Si esegue con `python3 scripts/03_10_modello_completo.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/03_10_modello_completo.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import math

import torch

from llm_anatomy.config import AgoGPTConfig
from llm_anatomy.model import AgoGPT
from llm_anatomy.utils import (conta_parametri, prepara, stampa_shape,
                               tabella, titolo, verifica)

prepara()

cfg = AgoGPTConfig(vocab_size=93, block_size=128, n_layer=4, d_model=192, n_head=6)
modello = AgoGPT(cfg).eval()
B, T = 2, 16
idx = torch.randint(0, cfg.vocab_size, (B, T))

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


In [3]:
titolo("Che modello e'")
print(f"  {cfg.descrizione()}")
print(f"  parametri: {conta_parametri(modello)}")


Che modello e'
--------------
  L=4 d=192 h=6/6kv d_head=32 d_ff=512 T=128 norm=rms act=swiglu pos=rope
  parametri: 1789056


In [4]:
titolo("Le forme, dall'inizio alla fine")
stampa_shape("indici in ingresso", idx)
with torch.no_grad():
    emb = modello.tok_emb(idx)
    stampa_shape("dopo l'embedding", emb)
    h = emb
    for i, blocco in enumerate(modello.blocchi):
        h, _ = blocco(h, modello.rope_cos[:T], modello.rope_sin[:T])
        stampa_shape(f"dopo il blocco {i}", h)
    hn = modello.norm_f(h)
    stampa_shape("dopo la norma finale", hn)
    logits = modello.lm_head(hn)
    stampa_shape("logit", logits)
print("  la forma non cambia mai dentro la pila: e' la stessa larghezza dal")
print("  primo blocco all'ultimo, ed e' cio' che rende sensato sommare i")
print("  contributi di blocchi diversi")


Le forme, dall'inizio alla fine
-------------------------------
  indici in ingresso           (2, 16)                int64
  dopo l'embedding             (2, 16, 192)           float32
  dopo il blocco 0             (2, 16, 192)           float32
  dopo il blocco 1             (2, 16, 192)           float32
  dopo il blocco 2             (2, 16, 192)           float32
  dopo il blocco 3             (2, 16, 192)           float32
  dopo la norma finale         (2, 16, 192)           float32
  logit                        (2, 16, 93)            float32
  la forma non cambia mai dentro la pila: e' la stessa larghezza dal
  primo blocco all'ultimo, ed e' cio' che rende sensato sommare i
  contributi di blocchi diversi


In [5]:
titolo("Rifatto a mano")
with torch.no_grad():
    logits_diretti, _ = modello(idx)
verifica(torch.allclose(logits, logits_diretti, atol=1e-5),
         "il percorso ricostruito passo per passo coincide con il forward")


Rifatto a mano
--------------
  [verifica superata] il percorso ricostruito passo per passo coincide con il forward


In [6]:
titolo("Appena costruito, non sa niente")
# Il bersaglio e' il token successivo, non il token stesso: con i pesi condivisi
# fra embedding e testa, predire se stessi sarebbe banale anche senza
# addestramento, e la perdita misurata non direbbe nulla.
with torch.no_grad():
    _, perdita = modello(idx[:, :-1], idx[:, 1:])
print(f"  perdita: {perdita.item():.4f}")
print(f"  log(V) = log({cfg.vocab_size}) = {math.log(cfg.vocab_size):.4f}")
verifica(abs(perdita.item() - math.log(cfg.vocab_size)) < 0.4,
         "il modello non addestrato e' vicino alla distribuzione uniforme")


Appena costruito, non sa niente
-------------------------------
  perdita: 4.5993
  log(V) = log(93) = 4.5326
  [verifica superata] il modello non addestrato e' vicino alla distribuzione uniforme


In [7]:
titolo("Una tabella usata due volte")
print(f"  tok_emb.weight e' lm_head.weight: "
      f"{modello.tok_emb.weight is modello.lm_head.weight}")
risparmio = cfg.vocab_size * cfg.d_model
print(f"  parametri risparmiati: {risparmio} su {conta_parametri(modello)} "
      f"({100 * risparmio / (conta_parametri(modello) + risparmio):.1f}%)")
print("  su un vocabolario da 50.000 simboli il risparmio diventa decisivo")


Una tabella usata due volte
---------------------------
  tok_emb.weight e' lm_head.weight: True
  parametri risparmiati: 17856 su 1789056 (1.0%)
  su un vocabolario da 50.000 simboli il risparmio diventa decisivo


In [8]:
titolo("Non sa nulla del futuro")
idx2 = idx.clone()
idx2[:, -1] = (idx2[:, -1] + 1) % cfg.vocab_size
with torch.no_grad():
    l1, _ = modello(idx)
    l2, _ = modello(idx2)
verifica(torch.allclose(l1[:, :-1], l2[:, :-1], atol=1e-6),
         "cambiare l'ultimo token non tocca le previsioni dei precedenti")
verifica(not torch.allclose(l1[:, -1], l2[:, -1], atol=1e-6),
         "ma cambia la previsione della posizione modificata")


Non sa nulla del futuro
-----------------------
  [verifica superata] cambiare l'ultimo token non tocca le previsioni dei precedenti
  [verifica superata] ma cambia la previsione della posizione modificata


In [9]:
titolo("Dove stanno i parametri")
gruppi = {
    "embedding (condivisi)": modello.tok_emb.weight.numel(),
    "attenzione": sum(conta_parametri(b.attn) for b in modello.blocchi),
    "FFN": sum(conta_parametri(b.ffn) for b in modello.blocchi),
    "norme": sum(conta_parametri(b.norm1) + conta_parametri(b.norm2)
                 for b in modello.blocchi) + conta_parametri(modello.norm_f),
}
totale = conta_parametri(modello)
righe = [[k, v, f"{100 * v / totale:.1f}%".replace(".", ",")]
         for k, v in gruppi.items()]
righe.append(["totale", totale, "100,0%"])
tabella(["gruppo", "parametri", "quota"], righe)
print("  la FFN pesa piu' dell'attenzione, con questo rapporto d_ff/d: e' vero")
print("  anche nei modelli grandi, dove le due parti stanno circa due a uno")


Dove stanno i parametri
-----------------------
gruppo                 parametri   quota
---------------------  ---------  ------
embedding (condivisi)      17856    1,0%
attenzione                589824   33,0%
FFN                      1179648   65,9%
norme                       1728    0,1%
totale                   1789056  100,0%
  la FFN pesa piu' dell'attenzione, con questo rapporto d_ff/d: e' vero
  anche nei modelli grandi, dove le due parti stanno circa due a uno


## Da provare

cambia n_layer e verifica che le tre verifiche passino comunque.